# Notebook 04 - Explanations: SHAP, Grad-CAM, attention and the segment-by-lead grid
Thesis Section 3.3.5, Table 3.1 and **Equation 2**.

| Method | Applied to | Output |
|---|---|---|
| SHAP (GradientExplainer) | 1D-CNN and CNN-BiLSTM | signed value per sample per lead |
| Grad-CAM | both (last convolutional layer) | one non-negative value per time point |
| Attention | CNN-BiLSTM only | one weight per time point |

Every map is converted to a **5 segment x 12 lead** matrix (P, PR, QRS, ST, T), using Equation 2: the sum of absolute attributions inside each segment of each lead, normalised so all 60 cells sum to one.

Explanations are made for every test record that truly has each superclass, using the seed-0 model of each architecture. GPU recommended; about 1-2 hours in total.

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

In [ ]:
from preprocessing.ptbxl import SUPERCLASSES, LEADS
from training.train import load_trained
from xai.shap_explainer import explain_shap
from xai.gradcam import explain_gradcam
from xai.attention import extract_attention
from xai.segment_alignment import batch_boundaries, align_batch, SEGMENTS
P = f"{DATA}/processed/ptbxl100"
X_test, y_test = np.load(f"{P}/X_test.npy"), np.load(f"{P}/y_test.npy")
X_train = np.load(f"{P}/X_train.npy", mmap_mode="r")
rng = np.random.default_rng(0)
background = np.asarray(X_train[np.sort(rng.choice(len(X_train), 20 if QUICK_TEST else 100, replace=False))])
models = {a: load_trained(a, 5, f"{RESULTS}/models/ptbxl_{a}/seed0")[0] for a in ["cnn1d", "cnn_bilstm_attention"]}
NSAMPLES = 10 if QUICK_TEST else 50        # SHAP samples per record; 50 is a good speed/quality balance on a T4
print(X_test.shape, background.shape)

## Step 4.1 - Segment boundaries for every test record
NeuroKit2 finds the P, QRS and T waves on lead II of each record (about 5 minutes). Where a wave cannot be found, a standard template around the R peak is used. The result is a "mask": a label 0-4 (P, PR, QRS, ST, T) for every sample.

In [ ]:
mask_file = f"{RESULTS}/segment_masks_test.npy"
if os.path.exists(mask_file):
    masks = np.load(mask_file)
else:
    masks = batch_boundaries(X_test, fs=100, lead=1)
    np.save(mask_file, masks)
print("records with at least one delineated beat:", int((masks >= 0).any(axis=1).sum()), "of", len(masks))
colors = ["#f4a261", "#e9c46a", "#e76f51", "#2a9d8f", "#264653"]
t = np.arange(1000) / 100
plt.figure(figsize=(12, 3)); plt.plot(t, X_test[0, 1], "k", lw=1)
for s in range(5):
    plt.fill_between(t, X_test[0, 1].min(), X_test[0, 1].max(), where=masks[0] == s, color=colors[s], alpha=0.3, label=SEGMENTS[s])
plt.legend(ncol=5); plt.title("Lead II with its five segments"); plt.xlabel("time (s)")
plt.savefig(f"{RESULTS}/figures/segments_example.png", dpi=300); plt.show()

### Optional: PTB-XL+ fiducial points (Section 3.3.5 names them as the first source)
Run the next cell once to see the PTB-XL+ ECGDeli files. If it prints P, QRS and T onset/offset symbols, you can switch to them with `xai.segment_alignment.segment_boundaries(signal, annotation=ann)`; NeuroKit2 stays the fallback. Report in Chapter 4 which source the boundaries came from.

In [ ]:
paths = json.load(open(f"{PROJECT}/paths.json"))
if "PTBXL_PLUS" in paths and os.path.exists(paths["PTBXL_PLUS"]):
    fid = sorted(glob.glob(f"{paths['PTBXL_PLUS']}/fiducial_points/ecgdeli/**/*", recursive=True))
    print(len(fid), "files, e.g.:", fid[:5])
else:
    print("PTB-XL+ not available in this session - skipped.")

## Step 4.2 - Compute the three kinds of maps and align them (Equation 2)

In [ ]:
grid_file = f"{RESULTS}/grids_test.npz"
store = dict(np.load(grid_file)) if os.path.exists(grid_file) else {}
for c, cls in enumerate(SUPERCLASSES):
    idx = np.where(y_test[:, c] == 1)[0]
    if QUICK_TEST:
        idx = idx[:8]
    store[f"idx__{cls}"] = idx
    X = X_test[idx]
    for arch, model in models.items():
        methods = {"shap": lambda: explain_shap(model, X, background, c, nsamples=NSAMPLES),
                   "gradcam": lambda: explain_gradcam(model, X, c)}
        if arch == "cnn_bilstm_attention":
            methods["attention"] = lambda: extract_attention(model, X)
        for method, fn in methods.items():
            key = f"{method}__{arch}__{cls}"
            if key in store:
                continue
            maps = fn()
            store[key] = align_batch(maps, masks[idx])
            if method == "shap":                                   # keep a few raw maps for figures
                store[f"raw_{key}"] = maps[:5]
            print(f"{cls:5s} {arch:22s} {method:9s} {len(idx):5d} records done")
    np.savez_compressed(grid_file, **store)
print("Saved", grid_file)

## Step 4.3 - Figure 4.4: one ECG with its SHAP map

In [ ]:
cls, arch = "MI", "cnn1d"
c = SUPERCLASSES.index(cls)
i = int(store[f"idx__{cls}"][0])
sv = store[f"raw_shap__{arch}__{cls}"][0]
lim = np.abs(sv).max()
fig, axes = plt.subplots(12, 1, figsize=(12, 16), sharex=True)
for l in range(12):
    ax = axes[l]
    ax.imshow(sv[l][None, :], aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim,
              extent=[0, 10, X_test[i, l].min(), X_test[i, l].max()], alpha=0.6)
    ax.plot(t, X_test[i, l], "k", lw=0.8); ax.set_ylabel(LEADS[l], rotation=0, labelpad=15); ax.set_yticks([])
axes[-1].set_xlabel("time (s)")
fig.suptitle(f"SHAP for {cls}, {arch} (red = pushes towards {cls}, blue = away)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/fig4_4_shap_example_{cls}.png", dpi=300); plt.show()

## Step 4.4 - Figure 4.5: segment-by-lead profiles by superclass (average 5 x 12 matrix)

In [ ]:
def profile(method, arch, cls):
    g = store[f"{method}__{arch}__{cls}"]
    valid = g.sum(axis=(1, 2)) > 0          # skip the few records where no beat could be delineated
    return g[valid].mean(axis=0)

combos = [("shap", "cnn1d"), ("gradcam", "cnn1d"), ("shap", "cnn_bilstm_attention"),
          ("gradcam", "cnn_bilstm_attention"), ("attention", "cnn_bilstm_attention")]
fig, axes = plt.subplots(len(combos), 5, figsize=(22, 3.2 * len(combos)))
for r, (method, arch) in enumerate(combos):
    for c, cls in enumerate(SUPERCLASSES):
        ax = axes[r, c]
        ax.imshow(profile(method, arch, cls), cmap="viridis", aspect="auto")
        ax.set_xticks(range(12), LEADS, fontsize=7); ax.set_yticks(range(5), SEGMENTS, fontsize=8)
        if r == 0: ax.set_title(cls)
        if c == 0: ax.set_ylabel(f"{method}\n{arch}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/fig4_5_profiles.png", dpi=300); plt.show()

## Step 4.5 - Table 4.6: share of attribution per segment, by class and method
Compare these with the established ECG criteria (e.g. MI: Q waves and ST/T changes in the territory leads; HYP: QRS voltage in V1-V6 and limb leads; CD: QRS width; STTC: ST and T).

In [ ]:
rows = []
for method, arch in combos:
    for cls in SUPERCLASSES:
        p = profile(method, arch, cls)
        rows.append({"Method": method, "Model": arch, "Class": cls,
                     **{s: round(float(p[k].sum()), 3) for k, s in enumerate(SEGMENTS)},
                     "Top leads": ", ".join(LEADS[j] for j in np.argsort(-p.sum(0))[:3])})
table46 = pd.DataFrame(rows)
table46.to_csv(f"{RESULTS}/tables/table4_6_segment_shares.csv", index=False); table46

**Done.** For Chapter 4, Section 4.4 keep Figure 4.4 (one per class if you like), Figure 4.5 and Table 4.6. Next: notebook 05.